In [ ]:
'''
    ### Main Goals of Notebook 03 - Model Architecture and Training

    1. **Introduction & Objectives**  
    Write a short introduction explaining the purpose of this notebook and how it 
    follows Notebook 02 (preprocessing is done, now we build and train the model).

    2. **Load the Data Generators**  
    Import and reload the train and validation generators created in Notebook 02. 
    This ensures the pipeline is connected and ready for training.

    3. **Build the Model Architecture**  
    Choose and create the model: Use Transfer Learning with ResNet50 (freeze base layers + add 
    custom classification head). This is recommended because it gives better 
    performance on medical images.

    4. **Compile the Model**  
    Set optimizer (Adam), loss (binary_crossentropy), and metrics (accuracy + AUC). 
    This prepares the model for effective training.

    5. **Add Training Callbacks**  
    Include ModelCheckpoint, EarlyStopping, and ReduceLROnPlateau. 
    These tools control training, prevent overfitting, and save the best model automatically.

    6. **Train the Model**  
    Train using the train generator, validation generator, and class weights (to handle imbalance). 
    Run for a reasonable number of epochs.

    7. **Visualize Training Results**  
    Plot accuracy, loss, and AUC curves for both training and validation sets. 
    This helps analyze if the model is learning well or overfitting.

    8. **Save the Model & Conclusion**  
    Save the best model file. Write a short conclusion with key choices, training results, 
    and next steps (evaluation + Grad-CAM in Notebook 04).
'''

In [ ]:
# 1 Introduction & Objectives
'''
    After completing the Exploratory Data Analysis in Notebook 01 and building a robust preprocessing 
    pipeline with data augmentation and class weights in Notebook 02, we now move to the core 
    phase of the project: model construction and training

    Notebook 01 revealed a significant class imbalance in the Chest X-Ray dataset (25.71% Normal vs 
                                                                                74.29% Pneumonia), 
    while Notebook 02 addressed this challenge through targeted data augmentation and calculated 
    class weights. The preprocessing pipeline is now fully validated, with images 
    resized to 224×224 and properly normalized.

    ## Objectives

    The main objectives of this notebook are:

    1. Design and implement a Deep Learning model using Transfer Learning (ResNet50) 
        adapted to the medical imaging task.
    2. Connect the model to the preprocessed data generators from Notebook 02.
    3. Train the model efficiently while handling class imbalance using the computed class weights.
    4. Monitor the training process with appropriate callbacks and visualize learning curves.
    5. Save the best performing model as a key deliverable for the PFA project.

    This notebook represents the modeling and training stage of our intelligent system for 
    automatic pneumonia detection from chest X-rays, as defined in the Cahier des Charges.
'''



In [ ]:
# 2 Loading the Data Generators

import tensorflow as tf
import albumentations
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

DATA_DIR = "../data/raw/chest_xray"


train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,              
    rotation_range=15,           
    width_shift_range=0.1,       
    height_shift_range=0.1,      
    zoom_range=0.12,             
    horizontal_flip=True,        
    fill_mode='nearest'          
)

# 2 Création des générateurs de données + Redimensionnement à 224x224
train_generator = train_datagen.flow_from_directory(
    directory=DATA_DIR + "/train",      
    target_size=(224, 224),             
    batch_size=32,
    class_mode='binary',                
    shuffle=True                       
)

class_weight = {
    0:1.9448,
    1:0.6730
}

Found 5216 images belonging to 2 classes.


In [ ]:
# 3 Building the Model Architecture

# Using Transfer Learning with ResNet50 (freeze base layers + custom classification head)

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# 1. Load ResNet50 base model (without top layers)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,     
    input_shape=(224, 224, 3)
)

# 2. Freeze the base layers
base_model.trainable = False

# 3. Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)        # Reduce spatial dimensions
x = Dropout(0.5)(x)                    # Regularization to prevent overfitting
x = Dense(128, activation='relu')(x)   # Small fully connected layer
output = Dense(1, activation='sigmoid')(x)   # Binary classification (Normal / Pneumonia)

# 4. Create the final model
model = Model(inputs=base_model.input, outputs=output)

model.summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)